# micrograd: backpropagation from scratch

In module 02 you called `loss.backward()` and gradients appeared. Here you build that machinery yourself. First comes a tiny autograd engine in about 100 lines of plain Python (no PyTorch, no NumPy) that trains a small neural network. Then you rewrite the same chain rule as matrix formulas, the form PyTorch actually runs, and train a classifier with gradients you derived on paper.

This is the most important module in the first half of the course. Once backprop stops being magic, every later topic (transformers, fine-tuning, LoRA) is engineering on top of something you already understand. If you only take one idea away: a neural network is a big math expression, and backpropagation is the chain rule applied to that expression, from the output backward.

### In this notebook

1. What a derivative is, numerically
2. A `Value` that remembers how it was made
3. Backprop by hand on a tiny expression
4. Automating it with `_backward` and a topological sort
5. One neuron, checked against PyTorch
6. A neural network library in 30 lines
7. Training it
8. Backprop with matrices: the four backward rules an MLP needs, derived and checked against autograd
9. Training a classifier with only your hand-written gradients

### How to work through it

Sections 1 to 7 follow Andrej Karpathy's video "The spelled-out intro to neural networks and backpropagation: building micrograd" (YouTube, about 2.5 hours). A good rhythm: watch a section, pause, write that part of the code in a blank cell yourself, then compare with the notebook. Sections 8 and 9 go past the video. Do them with paper and a pen next to you, because the derivations are the point.

### Before you start

- Finish module 02 first. You should have run `loss.backward()` and written a training loop.
- Have the video open in another window.
- No apps to run, nothing to download, and nothing new to install. Section 9's plots use matplotlib, which module 00 added. Every cell runs on the CPU in a few seconds.
- Plan on an afternoon or two: about 2.5 hours of video for sections 1 to 7, plus an hour or two for sections 8 and 9.
- Kernel: the `.venv` in `ML_Lab` (Select Kernel in the top right of VS Code).

## 1. What a derivative is, numerically

The derivative of `f` at `x` is how much `f` changes when you nudge `x` by a tiny amount `h`, divided by `h`. No calculus rules needed, just evaluate twice.

In [ ]:
def f(x):
    return 3*x**2 - 4*x + 5

h = 1e-6
x = 3.0
print("numerical slope at x=3:", (f(x + h) - f(x)) / h)
print("calculus says 6x - 4 =", 6*x - 4)

This works for any function, but it's far too slow for a network with millions of weights (one extra forward pass per weight) and it's imprecise. Backprop gets every derivative exactly in one backward pass. We'll keep the numerical version around as a checker.

## 2. A Value that remembers how it was made

The core object wraps a number, and records which values produced it and with what operation. Chain operations together and you get a graph of the whole computation.

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op="", label=""):
        self.data = data
        self.grad = 0.0              # d(output)/d(this value), filled in by backprop
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value({self.label}={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        return Value(self.data + other.data, (self, other), "+")

    def __mul__(self, other):
        return Value(self.data * other.data, (self, other), "*")


a = Value(2.0, label="a")
b = Value(-3.0, label="b")
c = Value(10.0, label="c")
e = a * b;  e.label = "e"
d = e + c;  d.label = "d"
f_ = Value(-2.0, label="f")
L = d * f_; L.label = "L"

print(L)
print("L was made by", L._op, "from", L._prev)

The graph is: `L = d * f`, `d = e + c`, `e = a * b`. Forward pass: `e = -6`, `d = 4`, `L = -8`.

## 3. Backprop by hand

Goal: `dL/dx` for every value. Start at the output and walk backward, using two local rules:

- For `z = x + y`, the gradient passes through unchanged: `dz/dx = 1`. Addition routes the gradient to both inputs.
- For `z = x * y`, each input's gradient is the other input: `dz/dx = y`. Multiplication swaps them.

The chain rule ties it together: `dL/dx = dL/dz * dz/dx`. The gradient arriving from above times the local derivative.

Work it out on paper before reading on:

| Value | Gradient | Why |
|---|---|---|
| L | 1 | dL/dL |
| d | -2 | L = d·f, so dL/dd = f = -2 |
| f | 4 | dL/df = d = 4 |
| c | -2 | d = e + c, plus routes it: -2 · 1 |
| e | -2 | same |
| a | 6 | e = a·b: dL/da = dL/de · b = -2 · -3 |
| b | -4 | dL/db = dL/de · a = -2 · 2 |

Now check every one numerically.

In [ ]:
def forward(a=2.0, b=-3.0, c=10.0, f=-2.0):
    e = a * b
    d = e + c
    return d * f

h = 1e-6
base = forward()
for name in ["a", "b", "c", "f"]:
    nudged = forward(**{name: {"a": 2.0, "b": -3.0, "c": 10.0, "f": -2.0}[name] + h})
    print(f"dL/d{name} ≈ {(nudged - base) / h:.4f}")

## 4. Automating it

Each operation knows its own local derivative. So when an operation creates a new Value, it also attaches a small function, `_backward`, that pushes the output's gradient into its inputs. Then `backward()` on the final output:

1. Sorts the graph so every node comes after everything it depends on (a topological sort).
2. Sets the output's gradient to 1.
3. Walks the nodes in reverse order, calling each one's `_backward`.

Gradients use `+=`, not `=`. If a value is used in two places, gradient flows back from both, and the contributions add (the multivariable chain rule). Forgetting this is a real, classic bug.

In [ ]:
import math

class Value:
    def __init__(self, data, _children=(), _op="", label=""):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, k):
        assert isinstance(k, (int, float)), "only constant powers"
        out = Value(self.data ** k, (self,), f"**{k}")
        def _backward():
            self.grad += k * self.data ** (k - 1) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")
        def _backward():
            self.grad += (1 - t**2) * out.grad      # d/dx tanh(x) = 1 - tanh(x)^2
        out._backward = _backward
        return out

    def exp(self):
        out = Value(math.exp(self.data), (self,), "exp")
        def _backward():
            self.grad += out.data * out.grad        # d/dx e^x = e^x
        out._backward = _backward
        return out

    # Everything else is built from the operations above.
    def __neg__(self):            return self * -1
    def __sub__(self, other):     return self + (-other)
    def __truediv__(self, other): return self * other**-1
    def __radd__(self, other):    return self + other       # lets you write 2 + Value
    def __rmul__(self, other):    return self * other       # lets you write 2 * Value
    def __rsub__(self, other):    return (-self) + other
    def __rtruediv__(self, other): return Value(other) * self**-1

    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

In [ ]:
a = Value(2.0); b = Value(-3.0); c = Value(10.0); f_ = Value(-2.0)
L = (a * b + c) * f_
L.backward()
for name, v in [("a", a), ("b", b), ("c", c), ("f", f_), ("L", L)]:
    print(name, v)

Same numbers as your hand calculation. Now the accumulation case: `b = a + a` should give `db/da = 2`.

In [ ]:
a = Value(3.0)
b = a + a
b.backward()
print("db/da =", a.grad, "(would be 1.0 if we'd used = instead of +=)")

## 5. One neuron, checked against PyTorch

A neuron takes inputs `x`, multiplies each by a weight `w`, adds a bias `b`, and squashes the result through an activation like `tanh`: `out = tanh(w1*x1 + w2*x2 + b)`.

The same neuron in micrograd and in PyTorch should give identical gradients.

In [ ]:
x1, x2 = Value(2.0), Value(0.0)
w1, w2 = Value(-3.0), Value(1.0)
b = Value(6.8813735870195432)
o = (x1*w1 + x2*w2 + b).tanh()
o.backward()
print("micrograd:", "o =", round(o.data, 4), "| grads w1, w2, x1, x2:",
      [round(v.grad, 4) for v in (w1, w2, x1, x2)])

import torch
tx1 = torch.tensor(2.0, dtype=torch.float64, requires_grad=True)
tx2 = torch.tensor(0.0, dtype=torch.float64, requires_grad=True)
tw1 = torch.tensor(-3.0, dtype=torch.float64, requires_grad=True)
tw2 = torch.tensor(1.0, dtype=torch.float64, requires_grad=True)
tb = torch.tensor(6.8813735870195432, dtype=torch.float64, requires_grad=True)
to = torch.tanh(tx1*tw1 + tx2*tw2 + tb)
to.backward()
print("pytorch:  ", "o =", round(to.item(), 4), "| grads w1, w2, x1, x2:",
      [round(t.grad.item(), 4) for t in (tw1, tw2, tx1, tx2)])

Notice `w2` has a gradient of 0. Its input `x2` is 0, so changing `w2` can't affect the output. PyTorch's autograd is your class scaled up: tensors instead of single numbers, many more operations, and kernels that run on GPUs.

## 6. A neural network library in 30 lines

In [ ]:
import random

class Neuron:
    def __init__(self, n_in, nonlin=True):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(n_in)]
        self.b = Value(0.0)
        self.nonlin = nonlin

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh() if self.nonlin else act

    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, n_in, n_out, **kwargs):
        self.neurons = [Neuron(n_in, **kwargs) for _ in range(n_out)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, n_in, n_outs):
        sizes = [n_in] + n_outs
        # every layer gets tanh except the last, which stays linear
        self.layers = [Layer(sizes[i], sizes[i+1], nonlin=(i != len(n_outs) - 1)) for i in range(len(n_outs))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

random.seed(1337)
model = MLP(3, [4, 4, 1])     # 3 inputs -> 4 -> 4 -> 1 output
print("parameters:", len(model.parameters()))
print("output on one input:", model([2.0, 3.0, -1.0]))

## 7. Training it

Four examples, each with 3 features and a target of +1 or -1. The network starts random. You'll run the same five-step loop from module 02, except now you built every piece of it.

In [ ]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]

random.seed(1337)
model = MLP(3, [4, 4, 1])
lr = 0.05

for step in range(100):
    preds = [model(x) for x in xs]                              # 1. forward
    loss = sum((p - y)**2 for p, y in zip(preds, ys))           # 2. loss
    for p in model.parameters():                                # 3. zero grads
        p.grad = 0.0
    loss.backward()                                             # 4. backward
    for p in model.parameters():                                # 5. update
        p.data -= lr * p.grad
    if step % 10 == 0:
        print(f"step {step:3}  loss {loss.data:.5f}")

print("\npredictions:", [round(p.data, 3) for p in preds])
print("targets:    ", ys)

The loss falls toward zero and the predictions end up close to the targets. That's a neural network learning, and you wrote every line of the math.

It's also painfully slow. Every multiply is a Python object with a closure attached, and a model with a billion weights would need billions of those objects per step. Real frameworks do the same math on whole matrices at once. So the next question is what the backward pass looks like when the unit of work is a matrix multiply instead of a single `*`.

## 8. Backprop with matrices

micrograd moves one number at a time. Real networks run matrix operations on whole batches, and PyTorch stores one backward formula per tensor operation (matrix multiply, add, tanh, softmax, and so on) instead of one per scalar. In this section you derive the four formulas a multilayer perceptron (MLP) needs, then check them against `loss.backward()`.

The network is the same kind of MLP as section 6's, written with matrices and ending in a classification output. A batch of $N$ examples, each with $D$ features, is stacked into a matrix $X$ of shape $(N, D)$, one example per row. It goes through one hidden layer of $H$ tanh units and an output layer that gives one score (a logit) for each of $C$ classes:

$$Z_1 = X W_1 + b_1, \qquad A_1 = \tanh(Z_1), \qquad Z_2 = A_1 W_2 + b_2, \qquad P = \mathrm{softmax}(Z_2), \qquad L = \text{cross-entropy}(P, y)$$

$W_1$ is $(D, H)$ and $W_2$ is $(H, C)$. $Z_1$ and $A_1$ (the hidden pre-activations and activations) are $(N, H)$. $Z_2$ (the logits) and $P$ (the class probabilities) are $(N, C)$. $y$ holds the true class of each example, and $L$ is one number, the loss averaged over the batch.

**Notation.** For any tensor $T$ in the forward pass, $dT$ is shorthand for $\partial L / \partial T$: how much the final loss changes when you nudge each entry of $T$. So $dT$ always has the same shape as $T$, one gradient entry per entry. That is the main shape-checking trick of this section. If a formula gives $dW$ a different shape than $W$, the formula is wrong.

Backprop runs the forward pass in reverse, and each step needs one of four rules.

### Rule 1: matrix multiply

Take $Y = XW$ with $X$ of shape $(N, D)$ and $W$ of shape $(D, H)$, so $Y$ has shape $(N, H)$. One entry of $Y$ is row $i$ of $X$ dotted with column $j$ of $W$:

$$Y_{ij} = \sum_{k=1}^{D} X_{ik} W_{kj}$$

Here $i$ picks the example (a row of the batch), $j$ picks the output unit, and $k$ runs over the input features.

Now ask which entries of $Y$ a single weight $W_{kj}$ touches. It shows up in $Y_{ij}$ for every row $i$ (always in column $j$), and its local derivative there is $\partial Y_{ij} / \partial W_{kj} = X_{ik}$, the number it was multiplied by. That's the multiplication rule from section 3. The chain rule adds up its effect through every entry it touched:

$$\frac{\partial L}{\partial W_{kj}} = \sum_{i=1}^{N} \frac{\partial L}{\partial Y_{ij}} \, \frac{\partial Y_{ij}}{\partial W_{kj}} = \sum_{i=1}^{N} dY_{ij} \, X_{ik}$$

The right side multiplies an entry of $dY$ indexed $(i, j)$ by an entry of $X$ indexed $(i, k)$ and sums over the shared index $i$. Compare that with how a matrix product works: $(AB)_{kj} = \sum_i A_{ki} B_{ij}$, where the summed index $i$ is the column of the first matrix and the row of the second. In $X_{ik}$ the index $i$ is the row, so flip $X$ first. The transpose $X^\top$ is $X$ with rows and columns swapped, $(X^\top)_{ki} = X_{ik}$, so the sum is $\sum_i (X^\top)_{ki} \, dY_{ij} = (X^\top dY)_{kj}$. For the whole matrix at once:

$$dW = X^\top dY$$

The same reasoning works for the input. $X_{ik}$ shows up in $Y_{ij}$ for every column $j$, with local derivative $W_{kj}$:

$$\frac{\partial L}{\partial X_{ik}} = \sum_{j=1}^{H} dY_{ij} \, W_{kj} = (dY \, W^\top)_{ik} \qquad\text{so}\qquad dX = dY \, W^\top$$

You don't have to memorize these, because shapes force them. $dW$ must have the shape of $W$, $(D, H)$, and its ingredients are $X$, shape $(N, D)$, and $dY$, shape $(N, H)$. The only way to multiply those two (transposing as needed) and land on $(D, H)$ is $X^\top dY$: $(D, N)(N, H) \to (D, H)$. Likewise $dX$ must be $(N, D)$, built from $dY$ $(N, H)$ and $W$ $(D, H)$, and only $dY \, W^\top$ fits: $(N, H)(H, D) \to (N, D)$. Get the shapes right and the matmul gradient comes out right.

The $\sum_i$ in $dW$ is section 4's `+=` rule. Each weight $W_{kj}$ is used once per row of the batch, $N$ times in all, and every use sends a gradient back. Those contributions add, and the matrix multiply $X^\top dY$ does the adding over the batch for you. That's why a weight's gradient accumulates over the whole batch.

### Rule 2: adding a bias

A layer adds a bias vector $b$ of shape $(H,)$ to every row: $Y = XW + b$, so $Y_{ij} = (XW)_{ij} + b_j$. PyTorch broadcasts $b$ across the $N$ rows (module 02). Addition passes the gradient through unchanged (section 3), and $b_j$ is used in all $N$ rows, so its contributions add:

$$\frac{\partial L}{\partial b_j} = \sum_{i=1}^{N} dY_{ij}$$

In code that's `db = dY.sum(0)`: sum over the batch dimension, which turns shape $(N, H)$ into $(H,)$, the shape of $b$. The general pattern to remember: broadcasting in the forward pass becomes a sum in the backward pass. Whenever one value gets copied to many places, its gradient is the sum of the gradients at all those places.

### Rule 3: elementwise activations

$A = \tanh(Z)$ applies tanh to each entry on its own. $A_{ij}$ depends only on $Z_{ij}$, so nothing is shared and nothing gets summed. Section 4's scalar tanh rule applies to every entry at once:

$$dZ = dA \odot (1 - A^2)$$

$\odot$ means elementwise multiplication (`*` on PyTorch tensors, as opposed to `@` for matrix multiply), and $A^2$ squares each entry. ReLU, $A = \max(0, Z)$, works the same way with its own local derivative, 1 where the input was positive and 0 elsewhere (you'll add ReLU to `Value` in Exercise 1):

$$dZ = dA \odot \mathbf{1}[Z > 0]$$

Here $\mathbf{1}[Z > 0]$ is a matrix with a 1 wherever $Z$ was positive and a 0 everywhere else, `(Z > 0)` in code.

### Rule 4: softmax plus cross-entropy

The output layer produces one row of $C$ logits $z = (z_1, \dots, z_C)$ per example. Logits are raw scores, any real numbers. Softmax turns them into probabilities that are positive and sum to 1:

$$p_j = \frac{e^{z_j}}{\sum_{k=1}^{C} e^{z_k}}$$

The cross-entropy loss for one example whose true class is $y$ is $-\log p_y$. It's near 0 when the model gives the true class a probability near 1, and it grows without limit as that probability goes to 0. Taking the log of the softmax splits the fraction into two terms:

$$\log p_y = z_y - \log \sum_{k=1}^{C} e^{z_k} \qquad\text{so}\qquad -\log p_y = -z_y + \log \sum_{k=1}^{C} e^{z_k}$$

Differentiate with respect to one logit $z_j$, one term at a time. The first term, $-z_y$, has derivative $-1$ if $j$ is the true class and $0$ otherwise. Write that as $-\mathbf{1}[j = y]$. For the second term, the derivative of $\log s$ is $1/s$, and the derivative of $s = \sum_k e^{z_k}$ with respect to $z_j$ is $e^{z_j}$. The chain rule multiplies them into $e^{z_j} / \sum_k e^{z_k}$, which is exactly $p_j$. Together:

$$\frac{\partial (-\log p_y)}{\partial z_j} = p_j - \mathbf{1}[j = y]$$

The loss is the mean over the batch, $L = \frac{1}{N} \sum_{n=1}^{N} \left(-\log p_{y_n}\right)$, where $y_n$ is the true class of example $n$. Each row's logits only affect that row's term, so every row gets the formula above, times $1/N$. As one matrix:

$$dZ = \frac{1}{N} \left(P - Y_{\text{onehot}}\right)$$

$P$ is the $(N, C)$ matrix of probabilities. $Y_{\text{onehot}}$ is an $(N, C)$ matrix with a 1 in each row at the true class and 0 everywhere else.

Read what this does to training. Gradient descent changes the weights so that each logit moves against its gradient. Every wrong class $j$ has gradient $p_j / N > 0$, so its logit gets pushed down in proportion to the probability the model wrongly gave it. The true class has gradient $(p_y - 1) / N < 0$, so its logit gets pushed up in proportion to $1 - p_y$, how far the model is from certain. A confident, correct prediction ($p_y \approx 1$, every other $p_j \approx 0$) gets almost no gradient. A confident, wrong one gets the most. Learning concentrates on the mistakes.

Why cross-entropy instead of squared error: minimizing the average $-\log p_y$ is the same as maximizing the product of the probabilities the model assigns to the true labels across the training set (the log turns that product into a sum), so training with it is maximum likelihood estimation.

### The code

The next three cells build a small random batch ($N = 8$ examples, $D = 4$ features, $H = 16$ hidden units, $C = 3$ classes) and a two-layer network. `mlp_forward` runs the forward pass and keeps every intermediate. `mlp_backward` uses exactly the four rules above, from the loss back to the input. Then the same network goes through autograd, with `F.cross_entropy` (softmax, log, and batch mean in one call), and the last cell prints the largest difference between each of your gradients and PyTorch's.

Two details. Everything is float64 on the CPU, so rounding error is around $10^{-16}$ and can't hide a real bug (Apple's MPS backend has no float64). And the softmax subtracts each row's largest logit before calling `exp`. That leaves $p$ unchanged, because the numerator and the denominator both get multiplied by the same factor $e^{-\max}$, but it keeps `exp` from overflowing on large logits.

Before you run the third cell, predict the shape of every gradient it prints.

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)
N, D, H, C = 8, 4, 16, 3     # batch size, input features, hidden units, classes

# float64 on the CPU, so a real bug can't hide inside rounding error
X = torch.randn(N, D, dtype=torch.float64)
y = torch.randint(0, C, (N,))                 # the true class of each example
params = {
    "W1": torch.randn(D, H, dtype=torch.float64) * 0.5,
    "b1": torch.randn(H, dtype=torch.float64) * 0.1,
    "W2": torch.randn(H, C, dtype=torch.float64) * 0.5,
    "b2": torch.randn(C, dtype=torch.float64) * 0.1,
}
print("X", tuple(X.shape), " y", y.tolist())
for name, t in params.items():
    print(name, tuple(t.shape))

In [ ]:
def mlp_forward(X, p):
    Z1 = X @ p["W1"] + p["b1"]                            # (N, H)
    A1 = torch.tanh(Z1)                                   # (N, H)
    Z2 = A1 @ p["W2"] + p["b2"]                           # (N, C) logits
    E = torch.exp(Z2 - Z2.max(dim=1, keepdim=True).values)  # shifting by the row max avoids overflow
    P = E / E.sum(dim=1, keepdim=True)                    # (N, C) probabilities, rows sum to 1
    return {"X": X, "Z1": Z1, "A1": A1, "Z2": Z2, "P": P}


def cross_entropy_loss(P, y):
    return -torch.log(P[torch.arange(len(y)), y]).mean()  # mean of -log p_y over the batch


def mlp_backward(cache, y, p):
    X, A1, P = cache["X"], cache["A1"], cache["P"]
    N, C = P.shape
    dZ2 = (P - F.one_hot(y, C).to(P.dtype)) / N   # rule 4: softmax + cross-entropy
    dW2 = A1.T @ dZ2                               # rule 1: (H, N) @ (N, C) -> (H, C)
    db2 = dZ2.sum(0)                               # rule 2: (N, C) -> (C,)
    dA1 = dZ2 @ p["W2"].T                          # rule 1: (N, C) @ (C, H) -> (N, H)
    dZ1 = dA1 * (1 - A1**2)                        # rule 3: tanh, elementwise
    dW1 = X.T @ dZ1                                # rule 1: (D, N) @ (N, H) -> (D, H)
    db1 = dZ1.sum(0)                               # rule 2: (N, H) -> (H,)
    dX = dZ1 @ p["W1"].T                           # rule 1: (N, H) @ (H, D) -> (N, D)
    return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2, "X": dX}

In [ ]:
# Your backward pass
cache = mlp_forward(X, params)
loss = cross_entropy_loss(cache["P"], y)
grads = mlp_backward(cache, y, params)

# The same network, with autograd doing the backward pass
leaves = {name: t.clone().requires_grad_() for name, t in params.items()}
X_leaf = X.clone().requires_grad_()
logits = torch.tanh(X_leaf @ leaves["W1"] + leaves["b1"]) @ leaves["W2"] + leaves["b2"]
loss_autograd = F.cross_entropy(logits, y)
loss_autograd.backward()

print(f"loss: manual {loss.item():.12f}   autograd {loss_autograd.item():.12f}\n")
print(f"{'tensor':<8}{'grad shape':<12}max |manual - autograd|")
for name in ["W1", "b1", "W2", "b2"]:
    diff = (grads[name] - leaves[name].grad).abs().max().item()
    print(f"{name:<8}{str(tuple(grads[name].shape)):<12}{diff:.2e}")
print(f"{'X':<8}{str(tuple(grads['X'].shape)):<12}{(grads['X'] - X_leaf.grad).abs().max().item():.2e}")

What to notice:

- Every difference is at the level of float64 rounding (around $10^{-17}$, and some print as exactly zero). Your four formulas and PyTorch's backward pass compute the same numbers.
- Every gradient has the shape of its tensor, which is the notation rule doing its job.
- Read `mlp_backward` again. Every line is a local gradient times the gradient arriving from above, exactly like micrograd's `_backward` closures. The only change is the unit of work, a whole matrix operation instead of one `*`. This is what `loss.backward()` does op by op for an MLP: at a matmul it runs two matmuls (one for $dW$, one for $dX$), at a broadcast add it sums, and at an elementwise op it multiplies by the local derivative.
- That pair of backward matmuls explains a number you'll see in scaling-law papers. Each forward matmul turns into two in the backward pass, so a training step costs about three forward passes. For a large model that comes to roughly 6 floating-point operations per parameter per training token: 2 forward, 4 backward.

## 9. Training with hand-written gradients

Now put your backward pass to work. The data below is a classic toy problem: three interleaved spirals in 2D, 100 points per class. No straight line separates the classes, so the network has to learn curved boundaries.

Training uses nothing but `mlp_forward`, `mlp_backward`, and plain gradient descent. Every parameter $\theta$ (each of $W_1, b_1, W_2, b_2$) takes the same step each iteration:

$$\theta \leftarrow \theta - \eta \, d\theta$$

$\eta$ (eta) is the learning rate. There's no autograd, no `requires_grad`, and no optimizer. Every step uses all 300 points, so this is full-batch gradient descent, and the gradient has none of the mini-batch noise from module 02, section 10.

The hidden layer has 100 tanh units. $W_1$ starts with standard normal entries, the biases start at zero, and $W_2$ is scaled by 0.01 so the first predictions are close to uniform across the three classes. Module 05, section 4 covers how to choose these scales on purpose.

Predict before you run the training cell: what should the loss be at step 0? (Hint: what is $-\log p_y$ when every class gets probability $1/3$?)

In [ ]:
import matplotlib.pyplot as plt

torch.manual_seed(0)
n_per_class, n_classes = 100, 3
points, labels = [], []
for k in range(n_classes):
    r = torch.linspace(0.0, 1.0, n_per_class, dtype=torch.float64)               # radius grows along the arm
    theta = torch.linspace(4.0 * k, 4.0 * (k + 1), n_per_class, dtype=torch.float64)
    theta = theta + 0.2 * torch.randn(n_per_class, dtype=torch.float64)          # a little noise
    points.append(torch.stack([r * torch.sin(theta), r * torch.cos(theta)], dim=1))
    labels.append(torch.full((n_per_class,), k))
X_spiral = torch.cat(points)      # (300, 2)
y_spiral = torch.cat(labels)      # (300,)
print("X_spiral", tuple(X_spiral.shape), " y_spiral", tuple(y_spiral.shape))

plt.figure(figsize=(4.5, 4.5))
plt.scatter(X_spiral[:, 0], X_spiral[:, 1], c=y_spiral, cmap="brg", s=14)
plt.title("Three-class spiral")
plt.gca().set_aspect("equal")
plt.show()

In [ ]:
import time

torch.manual_seed(0)
hidden = 100
spiral_params = {
    "W1": torch.randn(2, hidden, dtype=torch.float64),
    "b1": torch.zeros(hidden, dtype=torch.float64),
    "W2": torch.randn(hidden, n_classes, dtype=torch.float64) * 0.01,
    "b2": torch.zeros(n_classes, dtype=torch.float64),
}
lr, steps = 1.0, 3000

losses = []
start = time.time()
for step in range(steps):
    cache = mlp_forward(X_spiral, spiral_params)                 # forward
    losses.append(cross_entropy_loss(cache["P"], y_spiral).item())
    grads = mlp_backward(cache, y_spiral, spiral_params)         # your backward pass
    for name in spiral_params:
        spiral_params[name] -= lr * grads[name]                  # plain gradient descent
    if step % 500 == 0:
        print(f"step {step:5}  loss {losses[-1]:.4f}")

preds = mlp_forward(X_spiral, spiral_params)["P"].argmax(dim=1)
accuracy = (preds == y_spiral).double().mean().item()
print(f"final loss {losses[-1]:.4f}   accuracy {accuracy:.1%}   ({time.time() - start:.1f} s)")

In [ ]:
# Color a fine grid of points by the class the network predicts there
g = torch.linspace(-1.2, 1.2, 241, dtype=torch.float64)
gx, gy = torch.meshgrid(g, g, indexing="xy")
grid = torch.stack([gx.reshape(-1), gy.reshape(-1)], dim=1)
grid_class = mlp_forward(grid, spiral_params)["P"].argmax(dim=1).reshape(gx.shape)

fig, (ax_loss, ax_regions) = plt.subplots(1, 2, figsize=(11, 4.5))
ax_loss.plot(losses)
ax_loss.set_yscale("log")
ax_loss.set_xlabel("step")
ax_loss.set_ylabel("cross-entropy loss (log scale)")
ax_loss.set_title("Training loss")

ax_regions.contourf(gx, gy, grid_class, levels=[-0.5, 0.5, 1.5, 2.5], cmap="brg", alpha=0.25)
ax_regions.scatter(X_spiral[:, 0], X_spiral[:, 1], c=y_spiral, cmap="brg", s=14, edgecolors="k", linewidths=0.3)
ax_regions.set_aspect("equal")
ax_regions.set_title(f"Learned decision regions (accuracy {accuracy:.1%})")
plt.tight_layout()
plt.show()

What to notice:

- The loss starts at about 1.09, close to $\ln 3 \approx 1.10$, the loss of a model that gives every class probability $1/3$. It falls to about 0.03 in 3,000 steps, and the network classifies 99.3% of the points correctly in under a second on the CPU.
- For most of the first 900 steps the loss zigzags, rising slightly on about every other step, which is why the curve looks thick there. Each step overshoots the floor of a narrow valley in the loss surface and lands on the far side (module 02, section 9 has the geometry). It still converges. With `lr = 0.5` the curve is smooth, with no rises at all, but it ends a little higher (about 0.037 after 3,000 steps).
- The decision regions bend to follow the spiral arms. The tanh hidden layer is what makes curved boundaries possible. Without it, the two layers would collapse into a single linear layer, and every boundary would be a straight line.
- There's no "zero the gradients" step. `mlp_backward` returns fresh gradients on every call. Autograd needs `zero_grad()` only because it accumulates into `.grad` with `+=` (section 4).

That's a neural network learning, and you've now written every line of the math twice: once a number at a time with `Value`, once as matrix formulas. Everything bigger is the same idea with more engineering:

- Tensors batch thousands of scalar operations into one GPU kernel. Section 8's formulas are what those kernels compute on the way back.
- Better optimizers replace `p -= lr * grad`. You built momentum, Adam, and AdamW in module 02, section 11.
- Cross-entropy replaces squared error for classification. You derived its gradient in section 8 and trained with it here.
- Better architectures (transformers, module 06) replace the MLP. Their backward pass is still a chain of local rules like the four in section 8, plus a few more for attention and layer norm.

## Wrap-up

Frameworks compute gradients for you right up until a run produces NaNs, a loss refuses to move, or you need a custom layer. Then someone has to reason about gradients directly, and you want that person to be you. The same understanding lets you judge papers on their merits and estimate what a training run costs before you rent the GPUs (the 6 FLOPs per parameter per token from section 8). If you lead a company that builds on models, this is what lets you make real calls about your ML work instead of taking someone's word for it.

You're done when you can rebuild the `Value` class with `+`, `*`, `tanh`, and `backward()` in a blank file from memory, and derive $dW = X^\top dY$ and the softmax plus cross-entropy gradient on paper.

Going further:

- Karpathy's "Building makemore Part 4: Becoming a Backprop Ninja" (same YouTube series) does what section 8 does for a bigger network, including batch norm.
- Stanford CS231n's notes "Putting it together: Minimal Neural Network Case Study" train a small network on the same spiral data in NumPy.
- Module 05, section 4 picks up the next question: why the scale of the initial weights decides whether gradients survive a deep network.

---
# Exercises

**Exercise 1: Add ReLU.** ReLU is `max(0, x)` and is the most common activation in modern networks. Add a `relu()` method to `Value` with the correct `_backward` (what's the derivative when `x > 0`? when `x < 0`?). Verify against `torch.relu` for an input of `-2.0` and `3.0`.

**Exercise 2: A gradient checker.** Write `grad_check(fn, values, h=1e-6)` that takes a function building an expression from a list of floats, runs your backprop, computes numerical gradients by nudging each input, and prints both side by side. Test it on `lambda a, b, c: ((a * b).tanh() + c**2) / (a + 3)`.

**Exercise 3: Break it on purpose.** Rerun section 7's training loop with the "zero grads" step deleted. Watch the loss. Explain what's happening to the gradients each step.

**Exercise 4: Learning rate.** Using section 7's setup, train fresh models for 50 steps with `lr` set to 0.001, 0.05, and 0.5. Which barely moves, which works, and which blows up? Learning rate is the first thing you tune on any real model.

**Exercise 5: Deeper, with ReLU.** Extend section 8's network to two hidden layers with ReLU instead of tanh, $D \to H \to H \to C$, so six parameter tensors ($W_1, b_1, W_2, b_2, W_3, b_3$). Write `mlp2_forward` and `mlp2_backward`. Work out the backward pass on paper first, and keep the pre-activations $Z_1$ and $Z_2$ in the cache, since the ReLU rule needs them. Then verify every gradient (all six parameters plus $X$) against autograd on section 8's batch. Every max difference should be at float64 rounding level.

---
# Solutions

In [ ]:
# Exercise 1
def relu(self):
    out = Value(max(0.0, self.data), (self,), "relu")
    def _backward():
        self.grad += (1.0 if self.data > 0 else 0.0) * out.grad   # slope 1 if positive, 0 otherwise
    out._backward = _backward
    return out

Value.relu = relu   # attach the method to the existing class

for x in [-2.0, 3.0]:
    v = Value(x); out = v.relu(); out.backward()
    t = torch.tensor(x, requires_grad=True); torch.relu(t).backward()
    print(f"x={x}: micrograd grad {v.grad}, pytorch grad {t.grad.item()}")

In [ ]:
# Exercise 2
def grad_check(fn, values, h=1e-6):
    vals = [Value(v) for v in values]
    out = fn(*vals)
    out.backward()
    base = fn(*[Value(v) for v in values]).data
    for i, v in enumerate(values):
        nudged = list(values); nudged[i] = v + h
        numeric = (fn(*[Value(n) for n in nudged]).data - base) / h
        print(f"input {i}: backprop {vals[i].grad: .6f}   numeric {numeric: .6f}")

grad_check(lambda a, b, c: ((a * b).tanh() + c**2) / (a + 3), [0.5, -1.2, 2.0])

In [ ]:
# Exercise 3: without zeroing, each step's gradient is added on top of every previous step's.
# The effective update grows every step, so training overshoots and the loss stops falling smoothly
# (it may oscillate or blow up). This is exactly why PyTorch makes you call optimizer.zero_grad().
random.seed(1337)
model = MLP(3, [4, 4, 1])
for step in range(30):
    preds = [model(x) for x in xs]
    loss = sum((p - y)**2 for p, y in zip(preds, ys))
    loss.backward()                      # no zeroing
    for p in model.parameters():
        p.data -= 0.05 * p.grad
    if step % 3 == 0:
        print(f"step {step:3}  loss {loss.data:.5f}")

In [ ]:
# Exercise 4
# 0.001 learns slowly, 0.05 converges fast, 0.5 overshoots so badly the loss explodes (diverges).
for lr in [0.001, 0.05, 0.5]:
    random.seed(1337)
    model = MLP(3, [4, 4, 1])
    history = []
    for step in range(50):
        preds = [model(x) for x in xs]
        loss = sum((p - y)**2 for p, y in zip(preds, ys))
        for p in model.parameters():
            p.grad = 0.0
        loss.backward()
        for p in model.parameters():
            p.data -= lr * p.grad
        history.append(loss.data)
    print(f"lr={lr:<6} loss at steps 0/10/25/49: " + ", ".join(f"{history[i]:.4g}" for i in (0, 10, 25, 49)))

In [ ]:
# Exercise 5
# Two hidden layers with ReLU: D -> H -> H -> C. Same four rules, one more layer of them.
def mlp2_forward(X, p):
    Z1 = X @ p["W1"] + p["b1"];  A1 = torch.relu(Z1)
    Z2 = A1 @ p["W2"] + p["b2"]; A2 = torch.relu(Z2)
    Z3 = A2 @ p["W3"] + p["b3"]
    E = torch.exp(Z3 - Z3.max(dim=1, keepdim=True).values)
    P = E / E.sum(dim=1, keepdim=True)
    return {"X": X, "Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2, "P": P}


def mlp2_backward(cache, y, p):
    X, Z1, A1, Z2, A2, P = (cache[k] for k in ["X", "Z1", "A1", "Z2", "A2", "P"])
    N, C = P.shape
    dZ3 = (P - F.one_hot(y, C).to(P.dtype)) / N
    dW3, db3 = A2.T @ dZ3, dZ3.sum(0)
    dA2 = dZ3 @ p["W3"].T
    dZ2 = dA2 * (Z2 > 0)                 # ReLU passes gradient only where its input was positive
    dW2, db2 = A1.T @ dZ2, dZ2.sum(0)
    dA1 = dZ2 @ p["W2"].T
    dZ1 = dA1 * (Z1 > 0)
    dW1, db1 = X.T @ dZ1, dZ1.sum(0)
    dX = dZ1 @ p["W1"].T
    return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2, "W3": dW3, "b3": db3, "X": dX}


torch.manual_seed(1)
p2 = {
    "W1": torch.randn(D, H, dtype=torch.float64) * 0.5, "b1": torch.randn(H, dtype=torch.float64) * 0.1,
    "W2": torch.randn(H, H, dtype=torch.float64) * 0.5, "b2": torch.randn(H, dtype=torch.float64) * 0.1,
    "W3": torch.randn(H, C, dtype=torch.float64) * 0.5, "b3": torch.randn(C, dtype=torch.float64) * 0.1,
}
grads2 = mlp2_backward(mlp2_forward(X, p2), y, p2)

leaves2 = {name: t.clone().requires_grad_() for name, t in p2.items()}
X_leaf2 = X.clone().requires_grad_()
hidden2 = torch.relu(torch.relu(X_leaf2 @ leaves2["W1"] + leaves2["b1"]) @ leaves2["W2"] + leaves2["b2"])
F.cross_entropy(hidden2 @ leaves2["W3"] + leaves2["b3"], y).backward()

for name in p2:
    print(f"{name:<4}{str(tuple(grads2[name].shape)):<10}{(grads2[name] - leaves2[name].grad).abs().max().item():.2e}")
print(f"{'X':<4}{str(tuple(grads2['X'].shape)):<10}{(grads2['X'] - X_leaf2.grad).abs().max().item():.2e}")